In [1]:
import polars as pl

In [2]:
pl.__version__

'1.37.1'

In [4]:
# Read files
green_taxi_2025 = pl.read_parquet("data/green_tripdata_2025-11.parquet")

In [6]:
zones = pl.read_csv("data/taxi_zone_lookup.csv")

In [5]:
green_taxi_2025.head()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64
2,2025-11-01 00:34:48,2025-11-01 00:41:39,"""N""",1,74,42,1,0.74,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,0.0
2,2025-11-01 00:18:52,2025-11-01 00:24:27,"""N""",1,74,42,2,0.95,7.2,1.0,0.5,0.0,0.0,null,1.0,9.7,2,1,0.0,0.0
2,2025-11-01 01:03:14,2025-11-01 01:15:24,"""N""",1,83,160,1,2.19,13.5,1.0,0.5,5.0,0.0,null,1.0,21.0,1,1,0.0,0.0
2,2025-11-01 00:10:57,2025-11-01 00:24:53,"""N""",1,166,127,1,5.44,24.7,1.0,0.5,0.5,0.0,null,1.0,27.7,1,1,0.0,0.0
1,2025-11-01 00:03:48,2025-11-01 00:19:38,"""N""",1,166,262,1,3.2,18.4,3.75,1.5,1.0,0.0,null,1.0,24.65,1,1,2.75,0.0


In [7]:
zones.head()

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


In [8]:
zones.shape

(265, 4)

In [9]:
green_taxi_2025.shape

(46912, 21)

In [23]:
# Add column pickup datetime
green_taxi_2025 = green_taxi_2025.with_columns(
    pl.col("lpep_pickup_datetime").dt.date().alias("pickup_date")
)   


In [32]:
# Filter out rows with date 2025-12-01
green_taxi_2025 = green_taxi_2025.filter(
    pl.col("pickup_date") != pl.datetime(2025,12,1)
)

green_taxi_2025

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,pickup_date
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,date
2,2025-11-01 00:34:48,2025-11-01 00:41:39,"""N""",1,74,42,1,0.74,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,0.0,2025-11-01
2,2025-11-01 00:18:52,2025-11-01 00:24:27,"""N""",1,74,42,2,0.95,7.2,1.0,0.5,0.0,0.0,null,1.0,9.7,2,1,0.0,0.0,2025-11-01
2,2025-11-01 01:03:14,2025-11-01 01:15:24,"""N""",1,83,160,1,2.19,13.5,1.0,0.5,5.0,0.0,null,1.0,21.0,1,1,0.0,0.0,2025-11-01
2,2025-11-01 00:10:57,2025-11-01 00:24:53,"""N""",1,166,127,1,5.44,24.7,1.0,0.5,0.5,0.0,null,1.0,27.7,1,1,0.0,0.0,2025-11-01
1,2025-11-01 00:03:48,2025-11-01 00:19:38,"""N""",1,166,262,1,3.2,18.4,3.75,1.5,1.0,0.0,null,1.0,24.65,1,1,2.75,0.0,2025-11-01
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2025-11-30 19:58:34,2025-11-30 20:14:28,null,null,59,51,null,8.5,33.22,0.0,0.5,0.0,0.0,null,1.0,34.72,null,null,null,0.0,2025-11-30
2,2025-11-30 19:34:00,2025-11-30 19:46:00,null,null,74,151,null,1.73,13.86,0.0,0.5,0.77,0.0,null,1.0,16.13,null,null,null,0.0,2025-11-30
2,2025-11-30 21:46:46,2025-11-30 22:17:55,null,null,33,163,null,7.52,38.42,0.0,0.5,1.0,0.0,null,1.0,44.42,null,null,null,0.75,2025-11-30


## Question 3. Counting short trips 
#### For the trips in November 2025, how many trips had a trip_distance of less than or equal to 1 mile?


In [37]:
# Question 3. For the trips in November 2025, how many trips had a trip_distance of less than or equal to 1 mile?
green_taxi_2025.filter(pl.col("trip_distance") <= 1).select(pl.len())

len
u32
8007


## Question 4. Longest trip for each day
#### Which was the pick up day with the longest trip distance? Only consider trips with trip_distance less than 100 miles (to exclude data errors). Use the pick up time for your calculations.


In [62]:
green_taxi_2025 \
    .filter(pl.col("trip_distance") < 100) \
    .group_by("pickup_date") \
    .agg(
        pl.col("trip_distance").max().alias("max_trip_distance")
    ) \
    .sort("max_trip_distance", descending=True) \
    .head(1)


pickup_date,max_trip_distance
date,f64
2025-11-14,88.03


## Question 5. Biggest pickup zone

#### Which was the pickup zone with the largest total_amount (sum of all trips) on November 18th, 2025?

In [63]:
green_taxi_2025_joined = green_taxi_2025.join(zones, left_on="PULocationID", right_on="LocationID")
green_taxi_2025_joined

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,pickup_date,Borough,Zone,service_zone
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,date,str,str,str
2,2025-11-01 00:34:48,2025-11-01 00:41:39,"""N""",1,74,42,1,0.74,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone"""
2,2025-11-01 00:18:52,2025-11-01 00:24:27,"""N""",1,74,42,2,0.95,7.2,1.0,0.5,0.0,0.0,null,1.0,9.7,2,1,0.0,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone"""
2,2025-11-01 01:03:14,2025-11-01 01:15:24,"""N""",1,83,160,1,2.19,13.5,1.0,0.5,5.0,0.0,null,1.0,21.0,1,1,0.0,0.0,2025-11-01,"""Queens""","""Elmhurst/Maspeth""","""Boro Zone"""
2,2025-11-01 00:10:57,2025-11-01 00:24:53,"""N""",1,166,127,1,5.44,24.7,1.0,0.5,0.5,0.0,null,1.0,27.7,1,1,0.0,0.0,2025-11-01,"""Manhattan""","""Morningside Heights""","""Boro Zone"""
1,2025-11-01 00:03:48,2025-11-01 00:19:38,"""N""",1,166,262,1,3.2,18.4,3.75,1.5,1.0,0.0,null,1.0,24.65,1,1,2.75,0.0,2025-11-01,"""Manhattan""","""Morningside Heights""","""Boro Zone"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2025-11-30 19:58:34,2025-11-30 20:14:28,null,null,59,51,null,8.5,33.22,0.0,0.5,0.0,0.0,null,1.0,34.72,null,null,null,0.0,2025-11-30,"""Bronx""","""Crotona Park""","""Boro Zone"""
2,2025-11-30 19:34:00,2025-11-30 19:46:00,null,null,74,151,null,1.73,13.86,0.0,0.5,0.77,0.0,null,1.0,16.13,null,null,null,0.0,2025-11-30,"""Manhattan""","""East Harlem North""","""Boro Zone"""
2,2025-11-30 21:46:46,2025-11-30 22:17:55,null,null,33,163,null,7.52,38.42,0.0,0.5,1.0,0.0,null,1.0,44.42,null,null,null,0.75,2025-11-30,"""Brooklyn""","""Brooklyn Heights""","""Boro Zone"""


In [70]:
green_taxi_2025_joined \
    .filter(pl.col("pickup_date") == pl.datetime(2025,11,18)) \
    .group_by("Zone") \
    .agg(
        pl.col("fare_amount").sum().alias("amount")
    ) \
    .sort("amount", descending=True) \
    .head(1)

Zone,amount
str,f64
"""East Harlem North""",6555.32


## Question 6. Largest tip

#### For the passengers picked up in the zone named "East Harlem North" in November 2025, which was the drop off zone that had the largest tip? Note: it's tip , not trip. We need the name of the zone, not the ID.

In [75]:
harlem_north_pickups = green_taxi_2025_joined \
    .filter(pl.col("Zone") == "East Harlem North")

harlem_north_pickups.select(pl.len())

len
u32
12042


In [78]:
harlem_north_dropoffs = harlem_north_pickups.join(zones, left_on="DOLocationID", right_on="LocationID")
harlem_north_dropoffs

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,pickup_date,Borough,Zone,service_zone,Borough_right,Zone_right,service_zone_right
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,date,str,str,str,str,str,str
2,2025-11-01 00:34:48,2025-11-01 00:41:39,"""N""",1,74,42,1,0.74,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""Central Harlem North""","""Boro Zone"""
2,2025-11-01 00:18:52,2025-11-01 00:24:27,"""N""",1,74,42,2,0.95,7.2,1.0,0.5,0.0,0.0,null,1.0,9.7,2,1,0.0,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""Central Harlem North""","""Boro Zone"""
2,2025-11-01 00:38:13,2025-11-01 00:55:47,"""N""",1,74,262,1,3.4,19.8,1.0,0.5,2.0,0.0,null,1.0,27.05,1,1,2.75,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""Yorkville East""","""Yellow Zone"""
2,2025-11-01 00:20:44,2025-11-01 00:27:05,"""N""",1,74,74,1,0.78,7.9,1.0,0.5,0.0,0.0,null,1.0,10.4,1,1,0.0,0.0,2025-11-01,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""East Harlem North""","""Boro Zone"""
2,2025-10-31 23:58:40,2025-11-01 00:26:34,"""N""",1,74,265,1,16.25,61.8,1.0,0.5,1.0,0.0,null,1.0,65.3,1,1,0.0,0.0,2025-10-31,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Unknown""","""NA""","""N/A"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2025-11-30 16:50:21,2025-11-30 17:08:49,null,null,74,151,null,2.41,19.81,0.0,0.5,3.2,0.0,null,1.0,24.51,null,null,null,0.0,2025-11-30,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""Manhattan Valley""","""Yellow Zone"""
2,2025-11-30 16:15:00,2025-11-30 16:27:00,null,null,74,263,null,1.99,15.34,0.0,0.5,3.92,0.0,null,1.0,23.51,null,null,null,0.0,2025-11-30,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Manhattan""","""Yorkville West""","""Yellow Zone"""
2,2025-11-30 16:26:00,2025-11-30 16:44:00,null,null,74,167,null,2.75,8.22,0.0,0.5,0.0,0.0,null,1.0,9.72,null,null,null,0.0,2025-11-30,"""Manhattan""","""East Harlem North""","""Boro Zone""","""Bronx""","""Morrisania/Melrose""","""Boro Zone"""


In [80]:
harlem_north_dropoffs \
    .group_by("Zone_right") \
    .agg(
        pl.col("tip_amount").max().alias("max_tip_amount")
    ) \
    .sort("max_tip_amount", descending=True)

Zone_right,max_tip_amount
str,f64
"""Yorkville West""",81.89
"""LaGuardia Airport""",50.0
"""East Harlem North""",45.0
"""Long Island City/Queens Plaza""",34.25
"""NA""",28.9
…,…
"""South Ozone Park""",0.0
"""Gowanus""",0.0
"""Van Cortlandt Village""",0.0
